# Analyse univariée et bivariée des données
## TP fil rouge : Parcoursup 2025

**Données** : formations proposées sur Parcoursup en 2025 (ministère de l'Enseignement supérieur, data.enseignementsup-recherche.gouv.fr).

**Organisation** : en binôme, un dépôt GitHub par binôme. Un commit à la fin de chaque partie.

Pour chaque question : du code, **et** une interprétation rédigée. Un chiffre sans phrase ne vaut rien.

---

### 🎯 La question de l'enquête

> **Qu'est-ce qui rend une formation sélective sur Parcoursup ?**

La sélectivité se mesure avec `taux_acces_ens` : la part des candidats qui ont reçu une proposition d'admission. Plus il est **bas**, plus la formation est **sélective**.

Chaque partie du TP fait avancer l'enquête. À la fin de chaque jour, vous rédigez une synthèse : *ce qu'on sait à ce stade sur la sélectivité*. La conclusion finale fait partie du TP noté.

# J1 matin — Comprendre les fondements de l'univarié

## Mise en place

Créez votre dépôt GitHub, clonez-le, placez `parcoursup.csv` dans un dossier `data/`.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

Chargez le fichier (séparateur : `;`). Combien de lignes et de colonnes ?

In [3]:
#chargement des données
df = pd.read_csv('C:\\Users\\PC\\Documents\\python\\Analyse_univariee_bivariee\\parcoursup.csv', sep=';')
# Correction d'une faute de frappe dans l'en-tête du CSV : 'co=ntrat_etab' -> 'contrat_etab'
df = df.rename(columns={'co=ntrat_etab': 'contrat_etab'})
df.head()

C:\Users\PC\AppData\Local\Temp\ipykernel_16980\1412482743.py:2: DtypeWarning: Columns (110: detail_forma2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('C:\\Users\\PC\\Documents\\python\\Analyse_univariee_bivariee\\parcoursup.csv', sep=';')


,session,contrat_etab,cod_uai,g_ea_lib_vx,dep,dep_lib,region_etab_aff,acad_mies,ville_etab,lib_for_voe_ins,...,tri,cod_aff_form,detail_forma2,lien_form_psup,taux_acces_ens,part_acces_gen,part_acces_tec,part_acces_pro,etablissement_id_paysage,composante_id_paysage
0,2025,Public,0121471J,Institut national universitaire Champollion - ...,12,Aveyron,Occitanie,Toulouse,Rodez,Licence - Sciences et Techniques des Activités...,...,1_universités,2519,NaN,https://dossierappel.parcoursup.fr/Candidats/p...,7.0,93,5,2,NaN,NaN
1,2025,Privé enseignement supérieur,0753605L,FAC LIBRE THEOLOGIE PROTESTANT,75,Paris,Ile-de-France,Paris,Paris 14e Arrondissement,Licence - Théologie - Parcours Théologie prote...,...,1_universités,25320,NaN,https://dossierappel.parcoursup.fr/Candidats/p...,100.0,90,5,5,NaN,NaN
2,2025,Privé enseignement supérieur,0951809A,CY ILEPS,95,Val-d'oise,Ile-de-France,Versailles,Cergy,Licence - Sciences de l'éducation et de la for...,...,1_universités,25370,NaN,https://dossierappel.parcoursup.fr/Candidats/p...,77.0,80,16,5,NaN,NaN
3,2025,Public,0922671D,I.U.T de Ville d'Avray - Antenne de Nanterre,92,Hauts-de-Seine,Ile-de-France,Versailles,Nanterre,BUT - Techniques de commercialisation,...,1_universités,25438,NaN,https://dossierappel.parcoursup.fr/Candidats/p...,12.0,48,52,0,NaN,NaN
4,2025,Privé enseignement supérieur,0690195M,Institut Catholique de Lyon,69,Rhône,Auvergne-Rhône-Alpes,Lyon,Lyon 2e Arrondissement,Licence - Double diplôme - Licence Droit - Dou...,...,1_universités,25443,NaN,https://dossierappel.parcoursup.fr/Candidats/p...,60.0,100,0,0,NaN,NaN


## 1. Typologie des variables

🎯 *Enquête* : avant de chercher ce qui explique la sélectivité, il faut savoir quelles variables on a et comment les traiter.

Classez au moins ces colonnes : `fili`, `contrat_etab`, `select_form`, `cod_uai`, `capa_fin`, `voe_tot`, `taux_acces_ens`, `pct_bours`. Pour chacune : codage (ce que voit pandas), nature statistique, traitement retenu et justification.

| Variable | Codage | Nature statistique | Traitement retenu | Justification |
|---|---|---|---|---|
| | | | | |

In [4]:
variables = ["fili", "contrat_etab", "select_form", "cod_uai", "capa_fin", "voe_tot", "taux_acces_ens", "pct_bours"]
typologie = pd.DataFrame({
    "Variable": variables,
    "Codage pandas": [str(df[c].dtype) for c in variables],
    "Nature statistique": ["qualitative nominale", "qualitative nominale", "qualitative nominale binaire",
        "qualitative nominale (identifiant)", "quantitative discrete", "quantitative discrete",
        "taux arrondi (discret observe)", "taux arrondi (discret observe)"],
    "Traitement retenu": ["effectifs et barres", "effectifs et regroupement public/prive", "comparaison de groupes",
        "identification, sans moyenne", "continu pour distribution et correlations", "continu et logarithme",
        "continu, cible de selectivite", "continu, profil des admis"],
    "Justification": ["categories sans ordre", "statuts sans ordre", "deux categories sans ordre",
        "code administratif, sans distance numerique", "nombre entier de places, nombreuses valeurs",
        "nombre entier de voeux, nombreuses valeurs", "pourcentage entre 0 et 100",
        "pourcentage entre 0 et 100, non defini si aucun admis"]
})
typologie

,Variable,Codage pandas,Nature statistique,Traitement retenu,Justification
0,fili,str,qualitative nominale,effectifs et barres,categories sans ordre
1,contrat_etab,str,qualitative nominale,effectifs et regroupement public/prive,statuts sans ordre
2,select_form,str,qualitative nominale binaire,comparaison de groupes,deux categories sans ordre
3,cod_uai,str,qualitative nominale (identifiant),"identification, sans moyenne","code administratif, sans distance numerique"
4,capa_fin,int64,quantitative discrete,continu pour distribution et correlations,"nombre entier de places, nombreuses valeurs"
5,voe_tot,int64,quantitative discrete,continu et logarithme,"nombre entier de voeux, nombreuses valeurs"
6,taux_acces_ens,float64,taux arrondi (discret observe),"continu, cible de selectivite",pourcentage entre 0 et 100
7,pct_bours,float64,taux arrondi (discret observe),"continu, profil des admis","pourcentage entre 0 et 100, non defini si aucu..."


> 💾 **Commit** : « Typologie des variables »

## 2. Tendance centrale

🎯 *Enquête* : à quoi ressemble une formation « typique » ? Est-elle sélective ? On regarde d'abord la sélectivité, puis deux causes possibles : la demande (`voe_tot`) et la taille (`capa_fin`).

**2.1** Moyenne et médiane de `taux_acces_ens`. Que remarquez-vous ?

In [5]:
taux = df["taux_acces_ens"].dropna()
pd.Series({
    "moyenne": taux.mean(),
    "médiane": taux.median(),
    "valeurs manquantes": df["taux_acces_ens"].isna().sum()
})


moyenne               56.147363
médiane               54.000000
valeurs manquantes    15.000000
dtype: float64

**Interprétation** :

 moyenne **56,15 %**, médiane **54 %** (14 237 valeurs valides, 15 manquantes). La moyenne et la médiane sont proches : cela suggère peu d'asymétrie, **pas forcément une loi normale**. Pour décrire une formation typique, on retient la médiane : la moitié des formations a un taux d'accès inférieur ou égal à 54 %. Il s'agit de candidats ayant reçu une proposition, pas de candidats inscrits. **Cours, p. 11-13.**

**2.2** Moyenne et médiane de `voe_tot`. Une formation très demandée sera-t-elle forcément sélective ? Gardez la question en tête.

In [7]:
voeux = df["voe_tot"].dropna()
pd.Series({"moyenne": voeux.mean(), "médiane": voeux.median()})

moyenne    947.402821
médiane    385.000000
dtype: float64

**Interprétation** : 

moyenne **947,40 vœux**, médiane **385 vœux**. La moyenne dépasse nettement la médiane : quelques formations très demandées la tirent vers la droite. La médiane décrit mieux la formation typique ; la moyenne reste utile pour les totaux. Cette analyse univariée ne permet pas de conclure qu'une forte demande implique un faible taux d'accès : il faut aussi étudier la capacité. **Cours, p. 12-13.**

**2.3** Quelle formation a la plus grande capacité (`capa_fin`) ? Que deviennent la moyenne et la médiane sans elle ?

In [8]:
idx_max_capa = df["capa_fin"].idxmax()
formation_max_capa = df.loc[idx_max_capa, ["g_ea_lib_vx", "lib_for_voe_ins", "ville_etab", "capa_fin", "voe_tot", "taux_acces_ens"]]

comparaison_capa = pd.DataFrame({
    "avec toutes les formations": [df["capa_fin"].mean(), df["capa_fin"].median()],
    "sans la plus grande capacité": [df.drop(idx_max_capa)["capa_fin"].mean(), df.drop(idx_max_capa)["capa_fin"].median()]
}, index=["moyenne", "médiane"])

display(formation_max_capa)
display(comparaison_capa)


g_ea_lib_vx                                                     CNED
lib_for_voe_ins    BTS - Services - Diététique et nutrition - Ent...
ville_etab                                                  Grenoble
capa_fin                                                        3400
voe_tot                                                         1614
taux_acces_ens                                                 100.0
Name: 10282, dtype: object

,avec toutes les formations,sans la plus grande capacité
moyenne,53.981967,53.747176
médiane,30.000000,30.000000


**Interprétation** : 

 le **BTS Diététique et nutrition du CNED, entièrement à distance**, offre **3 400 places**. Sans cette formation, la moyenne des capacités passe de **53,98** à **53,75 places** ; la médiane reste **30 places**. La moyenne est sensible aux valeurs extrêmes, contrairement à la médiane, robuste. Le faible changement tient aussi aux 14 252 formations du fichier. Ce retrait est une démonstration, pas une suppression définitive. **Cours, p. 12-14.**

**2.4** Mode de `fili` et de `contrat_etab`.

In [9]:
display(df["fili"].value_counts().head(10))
display(df["contrat_etab"].value_counts())
print("Mode de fili :", df["fili"].mode().iloc[0])
print("Mode de contrat_etab :", df["contrat_etab"].mode().iloc[0])


fili
BTS                  5351
Licence              3052
Autre formation      1815
CPGE                  986
BUT                   820
Ecole d'Ingénieur     585
Licence_Las           513
IFSI                  344
PASS                  287
Ecole de Commerce     256
Name: count, dtype: int64

contrat_etab
Public                              11108
Privé sous contrat d'association     1938
Privé enseignement supérieur         1101
Privé hors contrat                    105
Name: count, dtype: int64

Mode de fili : BTS
Mode de contrat_etab : Public


**Interprétation** : 
 le mode de `fili` est **BTS** : **5 351 formations, soit 37,55 %**. Le mode de `contrat_etab` est **Public** : **11 108 formations, soit 77,94 %**. Le mode est adapté à ces variables nominales ; calculer une moyenne sur leurs codes n'aurait pas de sens. **Cours, p. 5 et 11-14.**

> 💾 **Commit** : « Tendance centrale »

## 3. Dispersion

🎯 *Enquête* : les formations sont-elles toutes à peu près aussi sélectives, ou y a-t-il de grands écarts à expliquer ?

On étudie la dispersion de `voe_tot` de bout en bout, puis on la compare à `taux_acces_ens`.

**3.1** Étendue de `voe_tot` (minimum, maximum, étendue).

In [10]:
pd.Series({
    "minimum": df["voe_tot"].min(),
    "maximum": df["voe_tot"].max(),
    "étendue": df["voe_tot"].max() - df["voe_tot"].min()
})


minimum        1
maximum    19404
étendue    19403
dtype: int64

**Interprétation** :minimum **1**, maximum **19 404**, étendue **19 403 vœux**. Cette grande étendue montre des situations extrêmes très différentes, mais elle repose uniquement sur deux valeurs et ne décrit pas la dispersion de l'ensemble. **Cours, p. 19.**

**3.2** Variance pas à pas sur les **10 premières formations** : écarts à la moyenne, écarts au carré, somme des écarts. Calculez la variance en divisant par n, puis par n − 1. Vérifiez avec numpy et pandas.

In [11]:
x = df["voe_tot"].dropna().head(10).reset_index(drop=True)
moyenne = x.mean()
table_variance = pd.DataFrame({
    "voe_tot": x,
    "écart à la moyenne": x - moyenne,
    "écart au carré": (x - moyenne) ** 2
})

variance_n = table_variance["écart au carré"].sum() / len(x)
variance_n_1 = table_variance["écart au carré"].sum() / (len(x) - 1)

print("Moyenne :", moyenne)
display(table_variance)
print("Somme des écarts :", table_variance["écart à la moyenne"].sum())
print("Variance divisée par n :", variance_n)
print("Variance divisée par n-1 :", variance_n_1)
print("Vérification numpy ddof=0 :", np.var(x))
print("Vérification numpy ddof=1 :", np.var(x, ddof=1))
print("Vérification pandas :", x.var())


Moyenne : 2499.4


,voe_tot,écart à la moyenne,écart au carré
0,1065,-1434.4,2057503.36
1,86,-2413.4,5824499.56
2,423,-2076.4,4311436.96
3,7012,4512.6,20363558.76
4,290,-2209.4,4881448.36
5,341,-2158.4,4658690.56
6,3502,1002.6,1005206.76
7,10913,8413.6,70788664.96
8,609,-1890.4,3573612.16
9,753,-1746.4,3049912.96


Somme des écarts : 4.547473508864641e-13
Variance divisée par n : 12051453.440000001
Variance divisée par n-1 : 13390503.822222224
Vérification numpy ddof=0 : 12051453.440000001
Vérification numpy ddof=1 : 13390503.822222224
Vérification pandas : 13390503.822222224


**Interprétation** : sur les 10 premières formations, la moyenne est **2 499,4**. La somme des écarts est nulle à l'arrondi numérique près ; celle des carrés vaut **120 514 534,4**. Division par `n=10` : **12 051 453,44 vœux²**. Division par `n-1=9` : **13 390 503,82 vœux²**. La seconde est plus grande d'un facteur 10/9. On divise par n pour décrire la série entière, par n-1 pour estimer la variance d'une population à partir d'un échantillon (correction de Bessel). Les 10 premières lignes ne sont pas un échantillon aléatoire représentatif. **Cours, p. 17-18.**

**3.3** Calculez tous les indicateurs de dispersion de `voe_tot` : variance, écart-type, écart absolu moyen, Q1, Q3, IQR, coefficient de variation. Comparez l'écart-type, l'écart absolu moyen et l'IQR.

In [12]:
def indicateurs_dispersion(s):
    s = s.dropna()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    return pd.Series({
        "variance": s.var(),
        "écart-type": s.std(),
        "écart absolu moyen": (s - s.mean()).abs().mean(),
        "Q1": q1,
        "Q3": q3,
        "IQR": q3 - q1,
        "coefficient de variation": s.std() / s.mean()
    })

indicateurs_dispersion(df["voe_tot"])


variance                    2.509634e+06
écart-type                  1.584182e+03
écart absolu moyen          9.298966e+02
Q1                          1.600000e+02
Q3                          1.016000e+03
IQR                         8.560000e+02
coefficient de variation    1.672132e+00
dtype: float64

**Interprétation** : variance **2 509 634,09 vœux²**, écart-type **1 584,18 vœux**, écart absolu moyen autour de la moyenne **929,90 vœux**. **Q1=160**, **Q3=1 016**, **IQR=856 vœux** : les 50 % centraux sont entre 160 et 1 016 vœux. Le **CV=1,672**, soit **167,21 %** de la moyenne, indique une forte dispersion relative. Les carrés donnent plus de poids aux extrêmes : l'écart-type dépasse l'écart absolu moyen. L'IQR est robuste et décrit le cœur de la distribution, mais ce n'est pas le même indicateur et on ne les compare pas comme des mesures interchangeables. **Cours, p. 17-20.**

**3.4** Comparez la dispersion de `voe_tot` et de `taux_acces_ens` (écart-type, IQR, coefficient de variation). Laquelle est la plus dispersée ? Qu'est-ce que ça dit des écarts de sélectivité entre formations ?

In [13]:
pd.DataFrame({
    "voe_tot": indicateurs_dispersion(df["voe_tot"]),
    "taux_acces_ens": indicateurs_dispersion(df["taux_acces_ens"])
}).loc[["écart-type", "IQR", "coefficient de variation"]]


,voe_tot,taux_acces_ens
écart-type,1584.182466,28.590671
IQR,856.000000,48.000000
coefficient de variation,1.672132,0.509208


**Interprétation** : `voe_tot` : écart-type **1 584,18 vœux**, IQR **856 vœux**, CV **167,21 %**. `taux_acces_ens` : écart-type **28,59 points de pourcentage**, IQR **48 points**, CV **50,92 %**. Les unités sont différentes : le plus grand écart-type brut ne suffit pas à trancher. Les CV, sans unité, montrent que la demande est plus dispersée **relativement à sa moyenne**. Les écarts de sélectivité restent importants : les 50 % centraux ont des taux entre **33 % et 81 %**. 

> 💾 **Commit** : « Dispersion »

## 4. Graphiques

🎯 *Enquête* : voir la sélectivité, et trouver un premier indice de ce qui la fait varier.

**4.1** Histogramme de `taux_acces_ens`. À quelle forme vous attendiez-vous ? Qu'obtenez-vous ?

**Interprétation** : _à rédiger_

**4.2** Diagramme en barres de `fili`, trié par effectif.

**4.2 bis** Camembert de `contrat_etab`, puis camembert de `fili`. Lequel est lisible ? Pourquoi ?

**Interprétation** : _à rédiger_

**4.3** Boxplot de `voe_tot`. Est-il lisible ? Que faudrait-il faire ?

**Interprétation** : _à rédiger_

**4.4** Boxplots de `taux_acces_ens` selon `select_form`. Premier indice : le label « sélective » correspond-il vraiment à un taux d'accès plus faible ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Graphiques », puis **push**.

# J1 après-midi — Approfondir les distributions

In [6]:
from scipy import stats

## 5. Asymétrie et aplatissement

🎯 *Enquête* : la forme des variables décidera des outils à utiliser au J2 pour les relier à la sélectivité.

**5.1** Calculez la skewness et la kurtosis de `voe_tot`, `taux_acces_ens`, `capa_fin` et `pct_bours`. Classez-les de la plus symétrique à la plus asymétrique. Lesquelles faudra-t-il transformer avant de les relier à la sélectivité ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Asymétrie »

## 6. La loi normale

🎯 *Enquête* : peut-on utiliser sur la sélectivité les outils qui supposent une loi normale ?

**6.1** Vérifiez la règle 68-95-99,7 sur `taux_acces_ens`. La variable suit-elle une loi normale ? Quelle conséquence pour la suite de l'enquête ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Loi normale »

## 7. Les distributions réelles

🎯 *Enquête* : la demande (`voe_tot`) est une cause possible de sélectivité. Quelle échelle faudra-t-il utiliser pour la comparer au taux d'accès ?

**7.1** Tracez l'histogramme de `voe_tot`, puis celui de `log(voe_tot)`. Quelle loi reconnaissez-vous ? Quelle échelle utiliserez-vous au J2 pour relier la demande à la sélectivité ?

**Interprétation** : _à rédiger_

**7.2** Quelle part des vœux concentrent les 10 % et les 20 % de formations les plus demandées ? Est-ce une distribution de type Pareto ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Distributions »

## 8. Outliers

🎯 *Enquête* : les formations hors norme risquent de fausser les liens avec la sélectivité. Faut-il les garder ?

**8.1** Détectez les outliers de `capa_fin` avec la méthode IQR, puis avec le Z-score (|z| > 3). Comparez les nombres obtenus et expliquez l'écart.

**Interprétation** : _à rédiger_

**8.2** Affichez les 10 plus grandes capacités. S'agit-il d'erreurs ou de valeurs réelles ? Que faites-vous ? Les garderez-vous pour étudier la sélectivité ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Outliers », puis **push**.

## 9. Analyse univariée en autonomie

🎯 *Enquête* : deux nouveaux suspects. Le profil social des admis et la région peuvent-ils jouer sur la sélectivité ?

Vous disposez maintenant de toute la méthode. Analysez **seuls** les deux variables suivantes, en suivant les étapes ci-dessous :
- une variable **quantitative** : `pct_bours` (part de boursiers parmi les admis, en %) ;
- une variable **qualitative** : `region_etab_aff` (région de l'établissement).

| Étape | Variable quantitative | Variable qualitative |
|---|---|---|
| 1. Comprendre | Ce qu'elle mesure, son unité, discrète ou continue | Nominale ou ordinale, liste des modalités |
| 2. Qualité | Valeurs manquantes, 0 suspects, bornes | Valeurs manquantes, modalités mal orthographiées |
| 3. Centre | Moyenne et médiane, comparées | Mode (et médiane si ordinale) |
| 4. Dispersion | σ, IQR, CV | Nombre de modalités, part de la modalité dominante |
| 5. Visualiser | Histogramme, boxplot | Diagramme en barres |
| 6. Forme | Skewness, kurtosis, QQ-plot : quelle loi ? | Répartition équilibrée ou concentrée, modalités rares |
| 7. Extrêmes | IQR ou Z-score selon la forme, puis inspection | — |
| 8. Conclure | Quel résumé, quels outils pour la suite ? | Quel regroupement, quel encodage pour la suite ? |

> Chaque chiffre s'interprète en une phrase.

### Variable quantitative : `pct_bours`

**Analyse** : _à rédiger_

### Variable qualitative : `region_etab_aff`

**Analyse** : _à rédiger_

> 💾 **Commit** : « Analyse univariée », puis **push**.

## 📝 Synthèse J1 — Ce qu'on sait sur la sélectivité

- À quoi ressemble la sélectivité d'une formation typique ? Les formations sont-elles proches ou très différentes ?
- Quels premiers indices avez-vous trouvés sur ce qui la fait varier ?
- Quelles variables faudra-t-il transformer pour la suite, et pourquoi ?

4 à 6 lignes.

**Synthèse** : _à rédiger_

> 💾 **Commit** : « Synthèse J1 », puis **push**.

# J2 matin — Mettre en pratique les fondements mathématiques de la corrélation

## 10. Covariance

🎯 *Enquête* : on passe au bivarié. On apprend d'abord à mesurer un lien sur un couple simple (capacité × vœux), avant de l'appliquer à la sélectivité.

**10.1** Tracez le nuage de points de `voe_tot` en fonction de `capa_fin`. Calculez leur covariance à la main (produits des écarts), puis avec pandas.

**Interprétation** : _à rédiger_

**10.2** Recalculez la covariance avec la capacité exprimée en centaines de places. Que constatez-vous ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Covariance »

## 11. Corrélation de Pearson

🎯 *Enquête* : première mesure chiffrée du lien entre la demande et la sélectivité.

**11.1** Calculez r entre `capa_fin` et `voe_tot` à la main (covariance / produit des écarts-types), puis avec pandas. Faites de même pour `voe_tot` et `taux_acces_ens`. Que dit ce second coefficient sur la sélectivité ?

**Interprétation** : _à rédiger_

**11.2** Les conditions de Pearson sont-elles respectées ? Recalculez r entre `log(capa_fin)` et `log(voe_tot)` (en excluant les valeurs nulles).

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Pearson »

## 12. Corrélation de Spearman

🎯 *Enquête* : le lien demande × sélectivité est-il sous-estimé par Pearson ?

**12.1** Calculez Spearman pour les deux couples de la question 11.1. Comparez avec Pearson et expliquez les écarts.

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Spearman »

## 13. Tests d'hypothèse

🎯 *Enquête* : le lien entre demande et sélectivité est-il réel, ou dû au hasard ? Et quel coefficient retenir pour la suite ?

**13.1** Testez la normalité de `taux_acces_ens` avec Shapiro-Wilk et Anderson-Darling, sur un échantillon de 50 formations puis sur toutes. Formulez H0 et concluez. Quel coefficient de corrélation retiendrez-vous pour la suite de l'enquête ?

**Interprétation** : _à rédiger_

**13.2** Testez la corrélation entre `voe_tot` et `taux_acces_ens` (Spearman). Formulez H0, donnez la p-value et concluez. Le lien est-il fort pour autant ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Tests », puis **push**.

# J2 après-midi — Approfondir les relations quantitatives

## 14. Corrélation partielle

🎯 *Enquête* : le lien entre demande et sélectivité tient-il à capacité égale ?

**14.1** Calculez la corrélation entre `log(voe_tot)` et `taux_acces_ens`, puis la corrélation partielle en contrôlant `log(capa_fin)`. Que concluez-vous ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Corrélation partielle »

## 15. A/B testing

🎯 *Enquête* : un lien observé suffit-il à affirmer qu'un facteur *cause* la sélectivité ?

**15.1** Au J3, vous verrez que les formations privées ont un taux d'accès plus élevé que les publiques. Peut-on en conclure que le statut privé *rend* une formation moins sélective ? Quelle expérience faudrait-il pour le prouver, et est-elle possible ici ?

**Interprétation** : _à rédiger_

**15.2** Reprenez la simulation de la démo avec 500 visiteurs, puis 20 000. Une version B qui fait passer le taux d'achat de 20 % à 22 %. L'effet est-il détecté dans les deux cas ? Pourquoi ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « A/B testing »

## 16. Matrice de corrélation

🎯 *Enquête* : vue d'ensemble. Quelles variables quantitatives sont liées à la sélectivité, et lesquelles disent la même chose ?

**16.1** Tracez la heatmap des corrélations de Spearman entre `capa_fin`, `voe_tot`, `acc_tot`, `taux_acces_ens`, `pct_bours`, `pct_f`, `pct_tb`. Quelles variables sont redondantes ? Quelles variables sont les meilleures candidates pour expliquer la sélectivité ?

**Interprétation** : _à rédiger_

## 17. Régression simple

🎯 *Enquête* : le niveau scolaire des admis explique-t-il la sélectivité, et dans quelle mesure ?

**17.1** Régressez `taux_acces_ens` sur `pct_tb` (part de mentions très bien parmi les admis). Donnez la pente, le R², et tracez les résidus. Les mentions TB expliquent-elles la sélectivité ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Régression », puis **push**.

## 📝 Synthèse J2 — Ce qu'on sait sur la sélectivité

- Quelles variables quantitatives sont liées à la sélectivité, avec quelle force ?
- Le lien avec la demande tient-il quand on contrôle la taille ?
- Peut-on parler de causes ? Pourquoi ?

4 à 6 lignes.

**Synthèse** : _à rédiger_

> 💾 **Commit** : « Synthèse J2 », puis **push**.

# J3 matin — Comprendre les fondements du bivarié qualitatif

## 18. Table de contingence et Chi²

🎯 *Enquête* : le label « formation sélective » dépend-il du statut de l'établissement ?

**18.1** Croisez `select_form` et `contrat_etab` : table des effectifs, puis fréquences conditionnelles par statut.

**Interprétation** : _à rédiger_

**18.2** Faites le test du Chi² d'indépendance (H0, effectifs théoriques, p-value), puis calculez le V de Cramér et les résidus standardisés.

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Chi² »

## 19. Quantitatif × qualitatif

🎯 *Enquête* : la sélectivité varie-t-elle selon le statut (public / privé) et selon la filière ?

**19.1** Le `taux_acces_ens` diffère-t-il entre formations publiques et privées ? Boxplot, Student (Welch) et Mann-Whitney.

**Interprétation** : _à rédiger_

**19.2** Le `taux_acces_ens` diffère-t-il selon la filière (`fili`) ? ANOVA et Kruskal-Wallis. Quelles filières sont les plus sélectives ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « Quanti × quali », puis **push**.

# J3 après-midi — Introduire le multivarié

## 20. ACP

🎯 *Enquête* : on rassemble toutes les variables. Où se place la sélectivité parmi elles ?

**20.1** Faites une ACP sur `log(capa_fin)`, `log(voe_tot)`, `taux_acces_ens`, `pct_bours`, `pct_f`, `pct_tb` (centrées-réduites). Quelle part de variance portent les deux premiers axes ? Interprétez-les à partir des contributions. Où se place `taux_acces_ens` sur ces axes ?

**Interprétation** : _à rédiger_

> 💾 **Commit** : « ACP », puis **push**.

## 🏁 Conclusion de l'enquête

Répondez à la question : **qu'est-ce qui rend une formation sélective sur Parcoursup ?**

- Les facteurs retenus, classés par importance, chacun appuyé par un chiffre du TP.
- Les pistes écartées.
- Les limites : ce que vos analyses ne permettent pas d'affirmer.

10 à 15 lignes. Cette conclusion compte dans la note du TP.

**Synthèse** : _à rédiger_

> 💾 **Commit** : « Conclusion de l'enquête », puis **push**.